In [97]:
#the pip commands to be used here

%pip install requests
%pip install pandas
%pip install seaborn


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [98]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path

import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [118]:
base_url = "http://192.168.43.25:8080/"


In [119]:
script_dir = Path().resolve().parent

def create_url(base_url,first_part_endpoint,second_part_endpoint):
    return base_url + first_part_endpoint + second_part_endpoint
def getData(url):
    response = requests.get(url)

    # Check response status
    if response.status_code == 200:
        data = response.json()
        print(f"Received {len(data)} records.")
        return data
    else:
        print(f"Error: {response.status_code}")

def saveDataIntoVariable(url,data_file_name):
    
    
    data = getData(url)
    if data:
       
        return pd.read_json(data)
    else:
        print("No data to save.")





In [120]:


def loadDataFromFile(file_name):
    script_dir = Path().resolve()
    file_path = script_dir / 'data' / (file_name + '.json')
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None

In [121]:
first_part_endpoint = "dataAnalysisEndpoints/"
second_part_endpoint = "getAllUserInputsExperimentState"        

url = create_url(base_url,first_part_endpoint,second_part_endpoint)

df_userInputData = saveDataIntoVariable(url,"UserInputs")

Received 83 records.


ValueError: Invalid file path or buffer object type: <class 'list'>

In [ ]:
df_userInputData

In [96]:
if 'details' in df_userInputData.columns:
    details_df = df_userInputData["details"].apply(pd.Series)
    
    # Then join with the original DataFrame (drop the nested 'details' if desired)
    df_userInputData = pd.concat([df_userInputData.drop(columns=["details"]), details_df], axis=1)

AttributeError: 'NoneType' object has no attribute 'columns'

In [ ]:
df_userInputData.columns

In [ ]:
if 0 in df_userInputData.columns:
    df_userInputData =df_userInputData.drop(columns=[0])

In [ ]:
# Convert epoch (assumes seconds) to datetime in local time
df_userInputData["epoch_ms"] = df_userInputData["timestamp"].apply(lambda x: x["$date"])
df_userInputData["timestamp_local"] = pd.to_datetime(df_userInputData["epoch_ms"], unit="ms").dt.tz_localize("UTC").dt.tz_convert("Europe/Athens")
df_userInputData.drop(columns = ["timestamp"])

In [ ]:
df_userInputData[df_userInputData["item-used"] == "Φαρμακευτικό αλκοόλ 95%"]

In [ ]:
# Define your filter condition
condition = df_userInputData["item-used"] == 'Φαρμακευτικό αλκοόλ 95%'
# Get the indexes where the condition is True
matching_idx = df_userInputData.index[condition]
# Also get the next index (i+1), and make sure it's within bounds
next_idx = matching_idx + 1
next_idx = next_idx[(next_idx < len(df_userInputData)) & (next_idx >= 0)]

# Combine the matching indexes with the next ones
final_idx = matching_idx.union(next_idx)

# Filter the DataFrame
df_userInputDataExperiments = df_userInputData.loc[final_idx]
df_userInputDataExperiments


In [ ]:
first_idx = df_userInputData.index[0]
last_idx = df_userInputData.index[-1]
start_time = df_userInputData.loc[first_idx,"epoch_ms"]
end_time   = df_userInputData.loc[last_idx,"epoch_ms"] 
first_part_endpoint = "timeSeriesEndpoints/"
second_part_endpoint = "getTimeSeriesData?start=" + str(start_time) + '&end=' +str(end_time)
url = create_url(base_url,first_part_endpoint,second_part_endpoint)
saveDataIntoDataFolder(url,file_name)


In [89]:
file_name="TimeSeries"

df_timeSeries = loadDataFromFile(file_name)
df_timeSeries

File /home/bob/Documents/Diploma project code/Diploma-Project/dataAnalysis and machine learning/data/TimeSeries.json does not exist.


In [90]:
# 1. Select only the needed column
df_timeSeriesDuringExperiments_copy = df_timeSeries[['timestamp', 'Id', 'BME680:breathVocEquivalent']].copy()

# 2. Convert `timestamp` (UTC) to Europe/Athens tz-aware datetime
df_timeSeriesDuringExperiments_copy['timestamp'] = (
    pd.to_datetime(df_timeSeriesDurinmogExperiments_copy['timestamp'], utc=True)
      .dt.tz_convert('Europe/Athens')
)
# 3. Create the three “Id=…:” columns
df_timeSeriesDuringExperiments_copy['Id=0:CCS811:TVOC'] = np.where(
    df_timeSeriesDuringExperiments_copy['Id'] == 0, df_timeSeriesDuringExperiments_copy['CCS811:TVOC'], np.nan
)
df_timeSeriesDuringExperiments_copy['Id=1:BME680:breathVocEquivalent'] = np.where(
    df_timeSeriesDuringExperiments_copy['Id'] == 1, df_timeSeriesDuringExperiments_copy['BME680:breathVocEquivalent'], np.nan
)
df_timeSeriesDuringExperiments_copy['Id=2:BME680:breathVocEquivalent'] = np.where(
    df_timeSeriesDuringExperiments_copy['Id'] == 2, df_timeSeriesDuringExperiments_copy['BME680:breathVocEquivalent'], np.nan
)

# 4. Set timestamp as the row index
df_timeSeriesDuringExperiments_copy.set_index('timestamp', inplace=True)

# 5. Drop the old Id+raw measurement columns
df_timeSeriesDuringExperiments_final = df_timeSeriesDuringExperiments_copy.drop(
    columns=['Id', 'BME680:breathVocEquivalent', 'CCS811:TVOC']
)

# Inspect result
df_timeSeriesDuringExperiments_final.head()
df_timeSeriesDuringExperiments_final

TypeError: 'NoneType' object is not subscriptable

In [91]:
df_timeSeriesDuringExperiments_final

NameError: name 'df_timeSeriesDuringExperiments_final' is not defined

In [92]:
# 1. Filter only rows 18 to 30 from df_userInputData
df_events = df_userInputData.iloc[19:30].reset_index(drop=True)
df_events


AttributeError: 'NoneType' object has no attribute 'iloc'

In [78]:
# 2. Identify experiment periods based on Inserting/RemovingSourcePollutant
experiment_periods = []
for i in range(0, len(df_events) - 1, 2):
    start_row = df_events.iloc[i]
    end_row = df_events.iloc[i + 1]
    
    if (start_row['experimentState'] == 'InsertingSourcePollutant' and
        end_row['experimentState'] == 'RemovingSourcePollutant'):

        start_time = pd.to_datetime(start_row['timestamp_local'])
        end_time = pd.to_datetime(end_row['timestamp_local'])
        experiment_periods.append((start_time, end_time))

# 3. Create plots using Seaborn
num_experiments = len(experiment_periods)
print (experiment_periods)


NameError: name 'df_events' is not defined

In [79]:
# Ensure the index is sorted and timezone-aware
df_timeSeriesDuringExperiments_final = df_timeSeriesDuringExperiments_final.sort_index()

# Find the actual bounds in the timeseries data for the experiment window
actual_start_time = df_timeSeriesDuringExperiments_final.index[
    df_timeSeriesDuringExperiments_final.index.searchsorted(start_time, side="left")
]

actual_end_time = df_timeSeriesDuringExperiments_final.index[
    df_timeSeriesDuringExperiments_final.index.searchsorted(end_time, side="right") - 1
]

# Slice the time series data between the actual bounds
df_period_data = df_timeSeriesDuringExperiments_final.loc[actual_start_time:actual_end_time]

NameError: name 'df_timeSeriesDuringExperiments_final' is not defined

In [80]:
fig, axes = plt.subplots(num_experiments, 2, figsize=(14, 4 * num_experiments), sharex=False)

# Ensure axes is always a 2D array
if num_experiments == 1:
    axes = np.array([axes])

for idx, (start, end) in enumerate(experiment_periods):
    # Slice time series data between start and end
    df_range = df_timeSeriesDuringExperiments_final.loc[start:end]

    # LEFT PLOT: Id=0:CCS811:TVOC
    sns.lineplot(
        ax=axes[idx, 0],
        data=df_range,
        x=df_range.index,
        y='Id=0:CCS811:TVOC',
        marker='o'
    )
    axes[idx, 0].set_title(f"data between {start} and {end}\nId=0:CCS811:TVOC")
    axes[idx, 0].set_xlabel("Timestamp")
    axes[idx, 0].set_ylabel("TVOC")

    # RIGHT PLOT: Id=1 and Id=2 for BME680
    sns.lineplot(
        ax=axes[idx, 1],
        data=df_range,
        x=df_range.index,
        y='Id=1:BME680:breathVocEquivalent',
        label="Id=1",
        marker='o'
    )
    sns.lineplot(
        ax=axes[idx, 1],
        data=df_range,
        x=df_range.index,
        y='Id=2:BME680:breathVocEquivalent',
        label="Id=2",
        marker='s'
    )
    axes[idx, 1].set_title(f"data between {start} and {end}\nBME680:breathVocEquivalent")
    axes[idx, 1].set_xlabel("Timestamp")
    axes[idx, 1].set_ylabel("VOC")
    axes[idx, 1].legend()

# Adjust layout
plt.tight_layout()
plt.show()

NameError: name 'num_experiments' is not defined